# Setup

In [1]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertModel
import gc

Google Colab specific. If local, set huggingface token with load from gitignore'd file

In [2]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
loading_file = False

## Configs

In [6]:
config = {
  "num_blocks":2,
  "rf_train":0.8,
  "text_batch_size":100,
  "rf_batch_size":100,
  "HRR_DIM": 1024,
  "alpha":(10/4), # bind sum weighing for combination
  "beta":(10/3), # bind sum weighing for originals
  "POOLS":[91,56,36], # based on wordnet synsets 91C3>117k, macroroles, and conceptnet relations
  "entropy_loss_weight":0.3,
  "MSE_loss_weight":1,
  "roleformer_hidden_dims":[100, 75],
  "epochs":100,
  "learning_rate":0.001
}

# "Learning with HRRs" Ops

In [7]:
seeded_np_gen = np.random.default_rng(332026)

In [8]:
# a = torch.randn((3, 4, 5))
# print(a.shape)
# a.shape[-1:]

In [ ]:
# Adapted from https://github.com/FutureComputing4AI/Learning-with-Holographic-Reduced-Representations/blob/main/LICENSE,
# https://github.com/MahmudulAlam/Holographic-Reduced-Representations/blob/main/HRR/with_pytorch.py
"""
Library functions to perform circular convolution operations.
"""

__author__ = "Ashwinkumar Ganesan, Sunil Gandhi, Hang Gao"
__email__ = "gashwin1@umbc.edu,sunilga1@umbc.edu,hanggao@umbc.edu"




"""
Numpy Functions.
"""
# Make them work with batch dimensions
# def cc(a, b):
#     return np.fft.irfft(np.fft.rfft(a) * np.fft.rfft(b))

# def np_inv(a):
#     return np.fft.irfft((1.0/np.fft.rfft(a)),n=a.shape[-1])

# def np_appx_inv(a):
#     #Faster implementation
#     return np.roll(np.flip(a, axis=-1), 1,-1)

def npcomplexMagProj(x):
    """
    Normalize a vector x in complex domain.
    """
    c = np.fft.rfft(x)

    # Look at real and image as if they were real
    c_ish = np.vstack([c.real, c.imag])

    # Normalize magnitude of each complex/real pair
    c_ish=c_ish/np.linalg.norm(c_ish, axis=0)
    c_proj = c_ish[0,:] + 1j * c_ish[1,:]
    return np.fft.irfft(c_proj,n=x.shape[-1])

# def nrm(a):
#     return a / np.linalg.norm(a)


def gen_rand_vec(dims, generator=None):
    """
    Generate a random vector of size dims.
    """
    if generator:
      out = generator.normal(loc=0, scale=1./dims, size=(dims))
      return npcomplexMagProj(out)
    else:
      return npcomplexMagProj(np.random.normal(0, 1. / dims, size=(dims)))

"""
Pytorch functions.
"""
# NB: these will be orthogonal to each other, but not to vecs from other function calls.
# def generate_vectors(num_vectors, dims):
#     """
#     Generate n vectors of size dims that are orthogonal to each other.
#     """
#     if num_vectors > dims:
#         raise ValueError("num_vectors cannot be greater than dims!")

#     # Intializing class vectors.
#     vecs = torch.randn(dims, num_vectors, dtype=torch.float)

#     # Using QR decomposition to get orthogonal vectors.
#     vecs, _ = torch.qr(vecs)
#     vecs = vecs.t()
#     vecs = vecs / torch.norm(vecs, dim=-1, keepdim=True)
#     return vecs

# def complex_multiplication(left, right):
#     """
#     Multiply two vectors in complex domain.
#     """
#     left_real, left_complex = left[..., 0], left[..., 1]
#     right_real, right_complex = right[..., 0], right[..., 1]

#     output_real = left_real * right_real - left_complex * right_complex
#     output_complex = left_real * right_complex + left_complex * right_real
#     return torch.stack([output_real, output_complex], dim=-1)

# def complex_division(left, right):
#     """
#     Divide two vectors in complex domain.
#     """
#     left_real, left_complex = left[..., 0], left[..., 1]
#     right_real, right_complex = right[..., 0], right[..., 1]

#     output_real = torch.div((left_real * right_real + left_complex * right_complex),(right_real**2 + right_complex**2))
#     output_complex = torch.div((left_complex * right_real - left_real * right_complex ),(right_real**2 + right_complex**2))
#     return torch.stack([output_real, output_complex], dim=-1)

def circular_conv(a, b):
    """ Defines the circular convolution operation
    a: tensor of shape (*, D)
    b: tensor of shape (*, D)
    """
    out_freq = torch.multiply(torch.fft.rfft(a, dim=-1), torch.fft.rfft(b, dim=-1))
    out = torch.fft.irfft(out_freq, n=a.shape[-1], dim=-1)
    return out

# def get_appx_inv(a):
#     """
#     Compute approximate inverse of vector a.
#     """
#     return torch.roll(torch.flip(a, dims=[-1]), 1,-1)
def approx_inverse(x):
    x = torch.flip(x, dims=[-1])
    return torch.roll(x, 1, dims=-1)

# def get_inv(a, typ=torch.DoubleTensor):
#     """
#     Compute exact inverse of vector a.
#     """
#     left = torch.rfft(a, 1, onesided=False)
#     complex_1 = np.zeros(left.shape)
#     complex_1[...,0] = 1
#     op = complex_division(typ(complex_1),left)
#     return torch.irfft(op,1,onesided=False)

# def complexMagProj(x):
#     """
#     Normalize a vector x in complex domain.
#     """
#     c = torch.rfft(x, 1, onesided=False)
#     c_ish=c/torch.norm(c, dim=-1,keepdim=True)
#     output = torch.irfft(c_ish, 1, signal_sizes=x.shape[1:], onesided=False)
#     return output

def normalize(x):
    return x/torch.norm(x)

# if do ortho=False, these will still be pseudo-orthogonal to each other
def get_vectors(num_vectors, dims, ortho=False, gen=None):
    # if ortho:
    #     vectors = generate_vectors(num_vectors, dims)
    #     return complexMagProj(vectors)
    # else:
    # TODO: vectorize with pytorch
    vectors = [gen_rand_vec(dims, gen) for i in range(num_vectors)]
    return torch.from_numpy(np.array(vectors, dtype=np.float32))


In [10]:
bases = get_vectors(10, 1024, gen=seeded_np_gen)
bases.requires_grad = False
bases01 = circular_conv(bases[0], bases[1])
print(bases01.shape)
print(approx_inverse(bases[1]).shape)
approx0 = circular_conv(bases01, approx_inverse(bases[1]))
print(approx0.shape)
torch.dot(approx0, bases[0]) # returns 1. so circular convolution doesn't blow
# up magnitude, and the approx inverse decodes perfectly (at least with one association)

torch.Size([1024])
torch.Size([1024])
torch.Size([1024])


tensor(1.0000)

# BERT

In [11]:
dbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-cased")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
dbert_model = DistilBertModel.from_pretrained("distilbert/distilbert-base-cased",
                                              output_attentions=True)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# print(sum([p.numel() for p in dbert_model.parameters()]))
# [p.dtype for p in dbert_model.parameters()]

In [14]:
dbert_model = dbert_model.to(device)

101 is [CLS], which is at the beginning. 102 is [SEP], which is at the end (or before the padding). 0 is padding

In [15]:
id2wordpart = {v:k for k, v in dbert_tokenizer.vocab.items()}

In [16]:
# # list(id2wordpart.values())[:100]
# print(len(set(dbert_tokenizer.vocab.values())))
# print(len(dbert_tokenizer.vocab.values()))
# # checked that there are no duplicate values - we're not missing anything when we invert the dict

In [17]:
# inp = dbert_tokenizer(["This is not a sentence.", "Or is it?"],
#                       padding=True,
#                       return_tensors='pt')
# print(inp)
# print(inp["input_ids"].shape)
# out = dbert_model.forward(inp["input_ids"])
# out.last_hidden_state.shape

In [18]:
def print_tokens(inp):
  assert len(inp.shape) == 1
  for id in inp:
    try:
      print(id2wordpart[id.item()])
    except:
      print(id.item())

In [19]:
# out1 = dbert_tokenizer(["This is not a sentence."],
#                       padding=True,
#                       return_tensors='pt')
# out2 = dbert_tokenizer(["A sentence is not what I am."],
#                       padding=True,
#                       return_tensors='pt')
# print(out1)
# print(out2)

In [20]:
# print_tokens(out1["input_ids"][0])
# print_tokens(out2["input_ids"][0])

In [21]:
# tests = ["bat the bats",
#          "cat the cats",
#          "obfusticator the obfusticators",
#          "zooble the zoobles",
#          "abcdefghijklmnopqrstuvwxyzmxdzklyscjaf",
#          "the is are was to of and a an back way give gave",
#          "describe guess choice source mom soon director",
#          "redhead despair pretentious disservice childlike complicit macaroni"]
# for T in tests:
#   print(f"{T}:")
#   out = dbert_tokenizer(T, return_tensors='pt')["input_ids"][0]
#   print_tokens(out)

In [22]:
# dbert_model

In [23]:
# obs = dbert_tokenizer(["obfusticator the obfusticators"], return_tensors='pt')["input_ids"]
# print_tokens(obs[0])
# obs = obs.to(device)
# dbert_model = dbert_model.to(device)
# out = dbert_model(obs)
# print(out.last_hidden_state.shape)
# print(out.last_hidden_state[0].shape)
# print(out.last_hidden_state[0][:,0])

## Merging words

In [24]:
# takes s-length input id vector and (s, 768) hidden state vector for sequence
# length s and merges together word part tokens into full words by adding their
# respective vectors together. returns an m-length tokens array and a (m,768)
# shaped tensor hidden state where m is the length of the merged tokens
def merge_tokens(input_ids, hidden_states):
  s = len(input_ids)
  out_toks = []
  out_vecs = []
  running_tok = ""
  running_vec = torch.zeros(768)#.to(device)
  vecs_added = 0
  for i in reversed(range(s)):
    tok = id2wordpart[input_ids[i].item()]
    if tok.startswith("##"):
      running_tok = tok.lstrip("##") + running_tok
      running_vec += hidden_states[i]
      vecs_added += 1
    else:
      if len(running_tok) > 0:
        # prepend this token to running and add to output list
        out_toks.append(tok + running_tok)
        # averaging so we hopefully maintain something like the layernorm at output of BERT
        vecs_added += 1
        # print(tok, vecs_added)
        out_vecs.append(1.0 / vecs_added * (running_vec + hidden_states[i]))
        # reset
        running_tok = ""
        running_vec = torch.zeros(768)
        vecs_added = 0
      else:
        # just add to list
        out_toks.append(tok)
        out_vecs.append(hidden_states[i])
  # reverse out lists
  out_toks.reverse()
  out_vecs.reverse()
  out_vecs = torch.vstack(out_vecs)
  return out_toks, out_vecs

In [25]:
# # print(1.0/5 * (0.5244 + 0.8159 - 0.0330 + 0.0982 - 0.2164)) # should match first elem of 2nd tok if averaging correctly
# merge_tokens(obs[0], out.last_hidden_state[0])

Freezing parameters for adaption.

In [26]:
for param in dbert_model.parameters():
  param.requires_grad = False

In [27]:
for name, param in dbert_model.named_parameters():
  print(name, param.requires_grad)

embeddings.word_embeddings.weight False
embeddings.position_embeddings.weight False
embeddings.LayerNorm.weight False
embeddings.LayerNorm.bias False
transformer.layer.0.attention.q_lin.weight False
transformer.layer.0.attention.q_lin.bias False
transformer.layer.0.attention.k_lin.weight False
transformer.layer.0.attention.k_lin.bias False
transformer.layer.0.attention.v_lin.weight False
transformer.layer.0.attention.v_lin.bias False
transformer.layer.0.attention.out_lin.weight False
transformer.layer.0.attention.out_lin.bias False
transformer.layer.0.sa_layer_norm.weight False
transformer.layer.0.sa_layer_norm.bias False
transformer.layer.0.ffn.lin1.weight False
transformer.layer.0.ffn.lin1.bias False
transformer.layer.0.ffn.lin2.weight False
transformer.layer.0.ffn.lin2.bias False
transformer.layer.0.output_layer_norm.weight False
transformer.layer.0.output_layer_norm.bias False
transformer.layer.1.attention.q_lin.weight False
transformer.layer.1.attention.q_lin.bias False
transforme

# Text Preprocessing

In [29]:
# with open("test_out.txt", "r") as fd:
#   lines = fd.readlines()
# print(len(lines))
# lines_split = [line.split(" ") for line in lines]
# print(max([len(L) for L in lines_split]))
# # print(lines[2])
# lines0 = dbert_tokenizer(lines[:3], padding=True, return_tensors='pt')
# # print(lines0)
# print_tokens(lines0["input_ids"][0])
# # lines0

In [30]:
# test_toks = dbert_tokenizer(lines, padding=True, return_tensors='pt')
# test_emb = dbert_model.forward(test_toks["input_ids"])

In [31]:
# dbert_tokenizer.pad([dbert_tokenizer(lines[0]), dbert_tokenizer(lines[1])],
#                     return_attention_mask=True)

In [32]:
# test_toks["input_ids"].dtype

In [33]:
# print(test_toks["input_ids"].shape)


In [34]:
# dbert_model.forward(test_toks["input_ids"][:3])

In [49]:
all_interviews = {}
all_lines = []
if not loading_file:
  fnames = os.listdir("/content/drive/MyDrive/ICCM26_Data/out_txt")
  # all_interviews = {}
  # all_lines = []
  for fname in fnames:
    with open(os.path.join("/content/drive/MyDrive/ICCM26_Data/out_txt", fname), "r") as fd:
      lines = fd.readlines()
    # strip newlines, remove empty strings
    lines = [L.rstrip("\n") for L in lines]
    lines = [L for L in lines if L != ""]
    all_interviews[fname] = lines
    all_lines.extend(lines)
    print("Lines:", len(lines), end=", ")
    lines_split = [line.split(" ") for line in lines]
    print("Max length:", max([len(L) for L in lines_split]))
  print(len(all_lines))
  all_lines[:10]

Lines: 576, Max length: 35
Lines: 608, Max length: 30
Lines: 484, Max length: 22
Lines: 489, Max length: 26
Lines: 513, Max length: 37
Lines: 356, Max length: 30
Lines: 509, Max length: 27
Lines: 894, Max length: 18
Lines: 465, Max length: 25
Lines: 465, Max length: 25
Lines: 622, Max length: 36
5981


## Datasets and Dataloaders

In [29]:
# for passing to BERT - just passing in raw tokens tensor takes far too long
class TextDataset(torch.utils.data.Dataset):
  # takes python array of lines of text
  def __init__(self, text):
    # eager loading, change to read files on demand if takes too much memory
    self.text = text
  def __getitem__(self, i):
    return dbert_tokenizer(self.text[i], padding=True)
  def __len__(self):
    return len(self.text)
  def collate_fn(self, batch):
    # print("raw batch passed in:", batch)
    B = len(batch)
    # print(f"batch size: {B}")
    lens = [len(ex["input_ids"]) for ex in batch]
    padded_batch = dbert_tokenizer.pad(batch, padding=True,
                                                 return_tensors='pt',
                                                 return_attention_mask=True)
    att_mask = padded_batch["attention_mask"]
    # print(type(att_mask))
    return padded_batch["input_ids"], att_mask, lens
# test_emb = dbert_model.forward(test_toks["input_ids"])

In [37]:
# text_dataset = TextDataset(lines)
# for line in text_dataset:
#   print(line)
#   break
# text_loader = torch.utils.data.DataLoader(
#     dataset = text_dataset,
#     batch_size = config["text_batch_size"],
#     shuffle = False,
#     collate_fn = text_dataset.collate_fn
# )
# for batch, att_mask, lens in text_loader:
#   print("Collated batch:", batch)
#   print(att_mask)
#   print(lens)
#   with torch.inference_mode():
#     out1 = dbert_model.forward(batch, att_mask)
#     out2 = dbert_model.forward(batch)
#   print(out1.last_hidden_state[0, :, 0])
#   print(out2.last_hidden_state[0, :, 0])
#   print(out1.last_hidden_state.shape)
#   break

Merging word parts and getting full list of tokens and vectors, to be treated as dataset for training role-former.

In [38]:
# all_merged_toks = []
# all_merged_vecs = []
# for batch, att_mask, lens in text_loader:
#   print("Collated batch:", batch)
#   print(att_mask)
#   print(lens)
#   with torch.inference_mode():
#     enc = dbert_model.forward(batch, att_mask)
#   # batch_merged_toks = []
#   # merged_vecs = []
#   for i in range(batch.size(0)):
#     # merge wordparts in each example
#     merged_toks, merged_vecs = merge_tokens(batch[i,:lens[i]],
#                                             enc.last_hidden_state[i,:lens[i],:])
#     if lens[i] > len(merged_toks):
#       print(f"Merging from {lens[i]} -> {len(merged_toks)} tokens")
#     all_merged_toks.append(merged_toks)
#     all_merged_vecs.append(merged_vecs)
#   break
# all_merged_toks, all_merged_vecs

In [39]:
# math.floor(config["rf_train"] * len(all_lines))

### Gather Full (skip if loading)

In [67]:
def ds_to_enc(text_dataset, symbols=False):
  full_text_loader = torch.utils.data.DataLoader(
      dataset = text_dataset,
      batch_size = config["text_batch_size"],
      shuffle = False,
      collate_fn = text_dataset.collate_fn
  )

  full_merged_toks = []
  full_merged_vecs = []
  all_words = set()
# for toks in all_merged_toks:
#   all_words.update(toks)
# all_words
# seeded_np_gen = np.random.default_rng(372026)

# # syms = get_vectors(len(all_words), dims=config["HRR_DIM"],gen=seeded_np_gen)

  for batch, att_mask, lens in full_text_loader:
    # print("Collated batch:", batch)
    # print(att_mask)
    # print(lens)
    with torch.inference_mode():
      batch = batch.to(device)
      att_mask = att_mask.to(device)
      enc = dbert_model.forward(batch, att_mask)
      enc.last_hidden_state = enc.last_hidden_state.to('cpu')
      batch = batch.to('cpu')

    # batch_merged_toks = []
    # merged_vecs = []
    for i in range(batch.size(0)):
      # merge wordparts in each example
      merged_toks, merged_vecs = merge_tokens(batch[i,:lens[i]],
                                              enc.last_hidden_state[i,:lens[i],:])
      # if lens[i] > len(merged_toks):
      #   print(f"Merging from {lens[i]} -> {len(merged_toks)} tokens")
      full_merged_toks.append(merged_toks)
      full_merged_vecs.append(merged_vecs)
      if symbols:
        all_words.update(merged_toks)
  if symbols:
    all_words = list(all_words)
    syms = get_vectors(len(all_words), dims=config["HRR_DIM"],gen=seeded_np_gen)
    word_to_symvec = {all_words[i]:syms[i] for i in range(len(all_words))}
    full_merged_syms = []
    for merged_toks in full_merged_toks:
      syms_i = [word_to_symvec[tok] for tok in merged_toks]
      syms_i = torch.vstack(syms_i) # S x N
      full_merged_syms.append(syms_i)
    return full_merged_toks, full_merged_vecs, full_merged_syms, word_to_symvec

  # full_merged_toks[0], full_merged_vecs[0]
  return full_merged_toks, full_merged_vecs

In [43]:
full_text_dataset_train, full_text_dataset_val, full_text_dataset_test = None, None, None
if not loading_file:
  full_text_dataset = None
  num_train = math.floor(config["rf_train"] * len(all_lines))
  num_val = num_train + math.floor((len(all_lines) - num_train)/2)
  full_text_dataset_train = TextDataset(all_lines[:num_train])
  full_text_dataset_val = TextDataset(all_lines[num_train:num_val])
  full_text_dataset_test = TextDataset(all_lines[num_val:])
  print(len(full_text_dataset_train), len(full_text_dataset_val), len(full_text_dataset_test))

In [42]:

ds_train_toks, ds_train_vecs = ds_to_enc(full_text_dataset_train)
ds_val_toks, ds_val_vecs = ds_to_enc(full_text_dataset_val)
ds_test_toks, ds_test_vecs = ds_to_enc(full_text_dataset_test)

AttributeError: 'NoneType' object has no attribute 'collate_fn'

In [ ]:
# all_words = set()
# for toks in all_merged_toks:
#   all_words.update(toks)
# all_words
# seeded_np_gen = np.random.default_rng(372026)

# # syms = get_vectors(len(all_words), dims=config["HRR_DIM"],gen=seeded_np_gen)


### PreProcessed Dataset

In [91]:
# Stores LLM encodings made from text, so we don't need to run
# inference every time we have a forward pass on the rest of the network
class PreProcessedTextDataset(torch.utils.data.Dataset):
  def __init__(self, vecs, syms=None):
    # self.toks = toks
    self.vecs = vecs
    self.syms = syms
  def __getitem__(self, i):
    if self.syms:
      return self.vecs[i], self.syms[i]
    return self.vecs[i]
  def __len__(self):
    return len(self.vecs)
  def collate_fn(self, batch):
    if not self.syms:
      B = len(batch)
      llm_enc_dtype = batch[0].dtype
      llm_enc_dim = batch[0].shape[1]
      lens = [ex.size(0) for ex in batch]
      max_len = max(lens)
      batch_padded = torch.zeros((B, max_len, llm_enc_dim),
                                dtype=llm_enc_dtype)
      batch_syms_padded = torch.zeros((B, max_len, llm_enc_dim),
                                dtype=llm_enc_dtype)
      att_mask = torch.zeros((B, max_len))

      for i, vec in enumerate(batch):
        if not self.syms:
          batch_padded[i,:lens[i],:] = vec
          att_mask[i,:lens[i]] = 1
      return batch_padded, att_mask, lens
    else:
      # vecs = [ex[0] for ex in batch]
      # syms = [ex[1] for ex in batch]
      # print(type(batch))
      # print(len(batch[0]), type(batch[0]))
      # print(len(batch[1]), type(batch[1]))
      # print(len(vecs), type(vecs[0]))
      # print(len(syms), type(syms[0]))
      # print(batch)
      B = len(batch)
      llm_enc_dtype = batch[0][0].dtype
      llm_enc_dim = batch[0][0].shape[1]
      sym_dim = batch[0][1].shape[1]
      lens = [ex[0].size(0) for ex in batch]
      max_len = max(lens)
      batch_vecs_padded = torch.zeros((B, max_len, llm_enc_dim),
                                dtype=llm_enc_dtype)
      batch_syms_padded = torch.zeros((B, max_len, sym_dim),
                                dtype=llm_enc_dtype)
      att_mask = torch.zeros((B, max_len))

      for i, (vec, sym) in enumerate(batch):
        batch_vecs_padded[i,:lens[i],:] = vec
        att_mask[i,:lens[i]] = 1
        batch_syms_padded[i, :lens[i], :] = sym
      return batch_vecs_padded, att_mask, lens, batch_syms_padded


In [ ]:
# test_pre_ds = PreProcessedTextDataset(all_merged_vecs)
# test_pre_dl = torch.utils.data.DataLoader(
#     dataset = test_pre_ds,
#     batch_size=config["rf_batch_size"],
#     num_workers=2,
#     shuffle=True, # change for training
#     collate_fn=test_pre_ds.collate_fn
# )
# for batch, att_mask, lens in test_pre_dl:
#   print(batch)
#   print(att_mask)
#   print(lens)
#   break

In [ ]:
full_pre_ds_train, full_pre_ds_val, full_pre_ds_test = None, None, None
if loading_file:
  full_pre_ds_train = torch.load("/content/drive/MyDrive/ICCM26_Data/full_pre_ds_train.pt", weights_only=False)
  full_pre_ds_val = torch.load("/content/drive/MyDrive/ICCM26_Data/full_pre_ds_val.pt", weights_only=False)
  full_pre_ds_test = torch.load("/content/drive/MyDrive/ICCM26_Data/full_pre_ds_test.pt", weights_only=False)
else:
  full_pre_ds_train = PreProcessedTextDataset(ds_train_vecs)
  full_pre_ds_val = PreProcessedTextDataset(ds_val_vecs)
  full_pre_ds_test = PreProcessedTextDataset(ds_test_vecs)
# full_pre_ds.collate_fn = PreProcessedTextDataset([]).collate_fn
full_pre_dl_train = torch.utils.data.DataLoader(
    dataset = full_pre_ds_train,
    batch_size=config["rf_batch_size"],
    num_workers=2,
    shuffle=True,
    collate_fn=full_pre_ds_train.collate_fn
)
full_pre_dl_val = torch.utils.data.DataLoader(
    dataset = full_pre_ds_val,
    batch_size=config["rf_batch_size"],
    num_workers=2,
    shuffle=True,
    collate_fn=full_pre_ds_val.collate_fn
)
full_pre_dl_test = torch.utils.data.DataLoader(
    dataset = full_pre_ds_test,
    batch_size=config["rf_batch_size"],
    num_workers=2,
    shuffle=True,
    collate_fn=full_pre_ds_test.collate_fn
)
for batch, att_mask, lens in full_pre_dl_train:
  print(batch)
  print(att_mask)
  print(lens)
  break

Store dataset for later

In [ ]:
if not loading_file:
  torch.save(full_pre_ds_train, '/content/drive/MyDrive/ICCM26_Data/full_pre_ds_train.pt')
  torch.save(full_pre_ds_val, '/content/drive/MyDrive/ICCM26_Data/full_pre_ds_val.pt')
  torch.save(full_pre_ds_test, '/content/drive/MyDrive/ICCM26_Data/full_pre_ds_test.pt')

# RoleFormer Architecture

In [32]:
class BindSum(nn.Module):
  def __init__(self, alpha=config["alpha"], beta=config["beta"]):
    super(BindSum, self).__init__()
    self.register_buffer("alpha", torch.tensor(alpha))
    self.register_buffer("beta", torch.tensor(beta))
  def forward(self, a, b):
    return 1.0/self.alpha * circular_conv(a, b) + 1.0/self.beta * (a + b)
# (*, HRR_DIM), (*, HRR_DIM) -> (*, HRR_DIM)
# def bind_sum(a, b, alpha=config["alpha"], beta=config["beta"]):
#   return 1.0/alpha * circular_conv(a, b) + 1.0/beta * (a + b)
BS = BindSum(config["alpha"], config["beta"])
BS(torch.randn(2, 3, 4, 5), torch.randn(2, 3, 4, 5)).shape


torch.Size([2, 3, 4, 5])

In [36]:
# # trying a faster way to do outer prods, adapted from https://discuss.pytorch.org/t/batch-outer-product/4025
# a = torch.randn((2, 3, 3))
# b = torch.randn((2, 3, 4))
# print("original a: \n", a)
# print("original b: \n", b)
# # a_reshaped = torch.reshape(a, (2*3, 3))
# # print("reshaped: \n", a_reshaped)
# # a_reshaped_back = torch.reshape(a_reshaped, (2, 3, 3))
# # print("reshaped back: \n", a_reshaped_back)
# a_reshape_op = a.reshape((2*3, 3)).unsqueeze(-1)
# print("reshaped a:\n", a_reshape_op, a_reshape_op.shape)
# b_reshape_op = b.reshape((2*3, 4)).unsqueeze(-2)
# print("reshaped b:\n", b_reshape_op, b_reshape_op.shape)
# out = torch.bmm(a_reshape_op, b_reshape_op).reshape((2, 3, 3, 4))
# print(out.shape)
# # now do manual outer
# outers = torch.zeros(2, 3, 3, 4)
# for i in range(2):
#   for j in range(3):
#     outers[i, j] = torch.outer(a[i, j], b[i, j])
# print(outers.shape)
# torch.all(out == outers) # returns true. so we can use the faster method


In [37]:
# c = torch.randn((4, 5))
# ri = out @ c
# print(ri.shape)
# # bound = circular_conv(ri, torch.randn(5))
# # print(bound.shape)
# circular_conv(ri, torch.randn(2, 3, 5).unsqueeze(-2)).shape
# # circular_conv(torch.zeros(2, 3, 5).unsqueeze(-1), torch.zeros(2, 3, 5, 5)).squeeze(-1).shape


In [38]:
# torch.nn.LogSoftmax()(torch.tensor([-1.0, 0.0, 3.0]))

In [39]:
# torch.randn((2, 3, 3, 5)).sum(dim=(-1, -2, -3)) / torch.tensor([2, 3])**2.0

In [40]:
# # # B, S, S, P masked by B, S
# a = torch.randn((2, 3, 3, 5))
# mask = torch.tensor([[0, 0, float("-inf")], [0, 0, 0]])
# print(a.shape, mask.shape)
# print(a)
# print(mask)
# # # mask.repeat_interleave(3, dim=0)
# mask1 = mask.unsqueeze(-1).unsqueeze(-1)
# mask2 = mask.unsqueeze(1).unsqueeze(-1)
# # print(mask2.shape)
# a_masked = a + mask1 + mask2
# print(a_masked[])
# F.softmax(a_masked, dim=-1)

In [41]:
# a = torch.tensor([0, 1, float("-inf")])
# print(F.softmax(a, dim=-1))
# print(nn.Softmax(dim=-1)(a))

## RoleFormer Class

In [34]:
# Inputs:
#   - tensor of role-sum HRR vector per token (batch_size, seq, HRR_DIM)
#   - at init: size of pool of new role vectors P
#   - at init: numpy generator for role pool, for reproduciblity
#   - mask vector for ignoring pad tokens
#  Output:
#   - role-sum HRR vector per token (batch_size, seq, HRR_DIM)
class RoleFormer(nn.Module):
  def __init__(self, role_pool_dim, rng, hrr_dim, hid_size,
               alpha=config["alpha"], beta=config["beta"]):
    super(RoleFormer, self).__init__()
    # Initialize role pool
    role_pool = get_vectors(role_pool_dim, hrr_dim, gen=rng)
    role_pool.requires_grad = False
    self.register_buffer("role_pool", role_pool)
    # self.role_pool = role_pool
    # init params
    P = role_pool_dim
    self.Wk = nn.Linear(hrr_dim, hid_size) #N->H
    self.Wq = nn.Linear(hrr_dim, hid_size)
    self.Wv = nn.Linear(hrr_dim, role_pool_dim)
    self.softmax = nn.Softmax(dim=-1)
    self.logsoftmax = nn.LogSoftmax(dim=-1) # for computing portion of loss
    self.bind_sum = BindSum(alpha, beta)

  # (B=batch_size, S=seq, N=HRR_DIM), (B,S), (B) -> (batch_size, seq, HRR_DIM), entropy
  def forward(self, Rs, mask, lens):
    K = self.Wk(Rs) # BxSxH
    Q = self.Wq(Rs) # BxSxH
    V = self.Wv(Rs) # BxSxP
    B, S, _ = K.shape
    _, _, P = V.shape
    KtQ = torch.bmm(Q, torch.transpose(K, 1, 2)) # BxSxS
    op1 = KtQ.reshape((B*S, S)).unsqueeze(-1) # (BxS)xSx1
    op2 = V.reshape((B*S, P)).unsqueeze(-2) # (BxS)x1xP
    # compute the outer product
    KtQv = torch.bmm(op1, op2).reshape((B, S, S, P))
    # apply additive mask where true tokens are 0 and padding tokens are -inf.
    # need to apply mask first to matching i's, then to matching j's in (:, i, j, :)
    # Mask doesn't work when doing binding after :(, so commenting out here
    # KtQv = KtQv + mask.unsqueeze(-1).unsqueeze(-1) + mask.unsqueeze(1).unsqueeze(-1)
    # the pairwise role weights
    R_nxt_wts_pairwise_unmasked = self.softmax(KtQv)#.nan_to_num(neginf=0.0)
    R_nxt_wts_pairwise_masked0 = R_nxt_wts_pairwise_unmasked * mask.unsqueeze(-1).unsqueeze(-1)
    R_nxt_wts_pairwise = R_nxt_wts_pairwise_masked0 * mask.unsqueeze(1).unsqueeze(-1)
    # print(R_nxt_wts_pairwise)
    # Take total entropy of role weights (before softmax for numerical
    # reasons), dividing by batch and true seq len
    # calc p*log(p) and turn -inf to 0
    logp = self.logsoftmax(KtQv)#.nan_to_num(neginf=0.0)
    plogp = torch.exp(logp) * logp
    # need to apply mask first to matching i's, then to matching j's in (:, i, j, :)
    # plogp *= mask.unsqueeze(-1).unsqueeze(-1)
    # plogp *= mask.unsqueeze(1).unsqueeze(-1)
    # print("plogp \n", plogp)
    # scale by maximum entropy
    avg_entropy_all = torch.sum(plogp, dim=-1) / torch.log(torch.tensor(P))#.unsqueeze(-1).unsqueeze(-1) # BSS
    # print(avg_entropy)
    avg_entropy_all_m = avg_entropy_all * mask.unsqueeze(-1)
    avg_entropy_batch = torch.sum(avg_entropy_all_m, dim=(-1, -2)) / (lens * S)#lens**2.0 # B
    avg_entropy = -1.0 / B * torch.sum(avg_entropy_batch) # scalar
    # the pairwise rolesum vectors
    RS_nxt_pairwise = R_nxt_wts_pairwise @ self.role_pool # BxSxSxN
    # BxSxSxN bind BxSx(1)xN -> BxSxSxN
    RS_nxt_overall = self.bind_sum(RS_nxt_pairwise, Rs.unsqueeze(-2))
    # need to apply mask first to matching i's, then to matching j's in (:, i, j, :)
    RS_nxt_overall_m1 = RS_nxt_overall * mask.unsqueeze(-1).unsqueeze(-1)
    RS_nxt_overall_m2 = RS_nxt_overall_m1 * mask.unsqueeze(1).unsqueeze(-1)
    RS_nxt_overall = torch.sum(RS_nxt_overall_m2, dim=-2) # BxSxN
    # Apply 1-0 mask at end to stop backprop to pad tokens
    # RS_nxt_overall *= mask.unsqueeze(-1)
    return RS_nxt_overall, avg_entropy


In [43]:
# import math
# for i in range(1, 10):
#   p = i/10.0
#   print(p, 1-p)
#   print(p * math.log(p) + (1-p) * math.log(1-p))
# math.log(2)

In [44]:
# a = F.log_softmax(torch.tensor([0.5, 0.5]), dim=-1)
# print(torch.sum(torch.exp(out) * out, dim=-1))
# out = F.log_softmax(torch.randn((2, 3, 3, 8)), dim=-1)
# torch.exp(out).sum(dim=-1)

In [45]:
# mask = torch.tensor([[1.0, 1, 0], [1.0, 1, 1]])
# print(mask)
# torch.randn((2, 3, 3, 8)) * mask.unsqueeze(-1).unsqueeze(-1) * mask.unsqueeze(1).unsqueeze(-1)
# torch.randn((2, 3, 8)) * mask.unsqueeze(-1)

In [46]:
# Yes, binding with zeros will give you random noise, not zero out anything
# bind_sum(torch.randn(5), torch.zeros(5))

In [35]:
test_roleformer = RoleFormer(5, np.random.default_rng(332026), 8, 10, config["alpha"], config["beta"])
Rs = torch.randn((2, 3, 8), generator=torch.manual_seed(352026))
mask = torch.tensor([[1.0, 1, 0], [1.0, 1, 1]])
test_roleformer(Rs, mask, torch.tensor([2, 3]))

(tensor([[[-0.1306, -1.2477,  1.1223, -1.0632,  0.5466, -0.5750,  1.3090,
           -0.5074],
          [-0.2884, -0.5245,  0.4113,  0.1540, -0.0199,  0.3493, -0.3497,
            0.3569],
          [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
            0.0000]],
 
         [[-2.5317,  0.3720, -0.1657,  0.3154, -1.2898,  0.2283, -1.3327,
           -0.6965],
          [ 1.5456, -0.5168,  0.8809, -0.4799,  0.5298, -0.2276, -0.5737,
           -1.1200],
          [ 1.7439, -0.2489,  1.7235,  0.7049,  1.2709,  1.2141,  0.6188,
           -0.6800]]], grad_fn=<SumBackward1>),
 tensor(0.9132, grad_fn=<MulBackward0>))

## RoleFormer Pipeline

In [126]:
# Inputs: Wordpart-merged BERT output
# Output: HRR rolesums (batch_size, seq, HRR_DIM)
class RoleFormerPipeline(nn.Module):
  def __init__(self, num_blocks, hidden_sizes, hrr_dim, pool0_dim,
               roleformer_pool_dims, bert_dim=768, rng=None, episodes=False):
    super(RoleFormerPipeline, self).__init__()
    self.FC = nn.Linear(bert_dim, pool0_dim)
    self.softmax = nn.Softmax(dim=-1)
    role_pool = get_vectors(pool0_dim, hrr_dim, gen=rng)
    self.register_buffer("role_pool", role_pool)
    assert len(hidden_sizes) == num_blocks == len(roleformer_pool_dims)
    role_formers = []
    for i in range(num_blocks):
      role_formers.append(RoleFormer(roleformer_pool_dims[i], rng, hrr_dim,
                                     hidden_sizes[i]))
    self.role_formers = nn.ModuleList(role_formers)
    self.hrr_dim = hrr_dim
    # if episodes:
    # Layer symbol vectors
    layer_symbs = get_vectors(num_blocks, hrr_dim, gen=rng)
    self.register_buffer("layer_symbs", layer_symbs)

  # input: (B, S, BERT), (B, S), (B,), (B, S, N) -> (B, S, N), (B, N), scalar
  def forward(self, x, mask, lens, episodes=False, symbs=None):
    # running multi-level representation HRR
    B = x.shape[0]
    N = self.hrr_dim
    multi = torch.zeros((B, N)).to(device)
    out1 = self.FC(x) # (B, S, P)
    out2 = self.softmax(out1) * mask.unsqueeze(-1)
    # Should I take entropy here too? it would have to be weighted less
    out3 = out2 @ self.role_pool # (B, S, N)
    total_entropy = 0.0
    for i, roleformer in enumerate(self.role_formers):
      out3, avg_entropy = roleformer(out3, mask, lens)
      total_entropy = total_entropy + avg_entropy
      if episodes:
        multi_out = circular_conv(out3, self.layer_symbs[i]).to(device) #BxSxN
        multi_out = circular_conv(multi_out, symbs) #BxSxN
        multi_out *= mask.unsqueeze(-1).to(device)
        multi += multi_out.sum(dim=(1)) #BxN
    total_entropy_out = total_entropy / len(self.role_formers)
    return out3, multi, total_entropy_out

In [128]:
gen = torch.manual_seed(352026)
llm_enc = torch.randn((2, 3, 768), generator=gen).to(device)
mask = torch.tensor([[1.0, 1, 0], [1.0, 1, 1]]).to(device)
lens = torch.tensor([2, 3]).to(device)
syms = torch.randn((2, 3, 8), generator=gen).to(device)
rfp = RoleFormerPipeline(num_blocks=2, hidden_sizes=[10,10], hrr_dim=8, pool0_dim=10,
                   roleformer_pool_dims=[9, 8]).to(device)
rfp.forward(llm_enc, mask, lens)
rfp.forward(llm_enc, mask, lens, episodes=True, symbs=syms)

(tensor([[[ 0.0217, -0.1619,  0.2544, -0.0877, -0.0164,  0.0636, -0.1384,
           -0.0544],
          [-0.0372, -0.1592,  0.2129, -0.1006,  0.0021,  0.0264, -0.0965,
           -0.0519],
          [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
            0.0000]],
 
         [[-0.1088, -0.3880,  0.5211, -0.1885, -0.0409,  0.2627, -0.3663,
            0.0349],
          [-0.0021, -0.4362,  0.5583, -0.2360, -0.0230,  0.1947, -0.3197,
           -0.0351],
          [ 0.1148, -0.2325,  0.4673, -0.1462,  0.0114,  0.0941, -0.1276,
           -0.0793]]], device='cuda:0', grad_fn=<SumBackward1>),
 tensor([[-0.9246,  0.1657, -0.2549, -1.4814,  1.0767, -0.5837,  0.5609,  1.0781],
         [-0.2752,  1.9941, -0.0867,  0.7575, -1.2186, -0.3666, -1.6162,  0.0971]],
        device='cuda:0', grad_fn=<AddBackward0>),
 tensor(0.9999, device='cuda:0', grad_fn=<DivBackward0>))

## Autoencoder

In [129]:
class RoleAutoEncoder(nn.Module):
  def __init__(self, **pipeline_kwargs):
    super(RoleAutoEncoder, self).__init__()
    # define RoleFormerPipeline obj
    self.role_former_pipeline = RoleFormerPipeline(**pipeline_kwargs)
    # define decoder
    self.FC = nn.Linear(pipeline_kwargs["hrr_dim"], pipeline_kwargs["bert_dim"])
    self.gelu = nn.GELU()
    self.layer_norm = nn.LayerNorm(pipeline_kwargs["bert_dim"])
  # should not call for episode=True! This doesn't return HRRs, just a
  # reconstruction of the LLM encodings
  def forward(self, **kwargs):
    # call RoleFormerPipeline
    out, _, entropy = self.role_former_pipeline(**kwargs)

    # call FC decoder
    out1 = self.FC(out)
    out2 = self.gelu(out1)
    out3 = self.layer_norm(out2)
    return out3, entropy

gen = torch.manual_seed(352026)
llm_enc = torch.randn((2, 3, 768), generator=gen)
mask = torch.tensor([[1.0, 1, 0], [1.0, 1, 1]])
lens = torch.tensor([2, 3])
syms = torch.randn((2, 3, 8), generator=gen)
rae = RoleAutoEncoder(num_blocks=2, hidden_sizes=[10,10], hrr_dim=8, pool0_dim=10,
                   roleformer_pool_dims=[9, 8], bert_dim=768)
rae.forward(x=llm_enc, mask=mask, lens=lens)#.shape
# rfp.forward(llm_enc, mask, lens, syms, episodes=True)

(tensor([[[ 0.1816, -0.5519, -1.3597,  ..., -0.9668, -0.3050, -0.0470],
          [ 0.1477, -0.6376, -1.3577,  ..., -0.9120, -0.2526, -0.0531],
          [ 0.1318, -0.6321, -1.0922,  ..., -1.1075, -0.0837, -0.0640]],
 
         [[-0.2157, -0.6601, -1.4918,  ..., -0.7830, -0.1615, -0.2229],
          [-0.0278, -0.6950, -1.4823,  ..., -0.7302, -0.2490, -0.1282],
          [ 0.0119, -0.7516, -1.4360,  ..., -0.4782, -0.5141,  0.0595]]],
        grad_fn=<NativeLayerNormBackward0>),
 tensor(0.9999, grad_fn=<DivBackward0>))

# Fast Weight Programmers

# Model Setup

In [121]:
def save_model(model, optimizer, epoch, loss, path):
    torch.save({
        'model_state_dict'         : model.state_dict(),
         'optimizer_state_dict'    : optimizer.state_dict(),
         'loss'                : loss,
         'epoch'                   : epoch},
         path)

def load_model(path, model, optimizer=None):

    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    out = (model,)
    if optimizer != None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        out = out + (optimizer,)
    loss  = checkpoint['loss']
    epoch   = checkpoint['epoch']
    out = out + (loss, epoch)
    return out

In [130]:
# gen = torch.manual_seed(352026)
seeded_np_gen = np.random.default_rng(382026)
rae = RoleAutoEncoder(num_blocks=config["num_blocks"],
                      hidden_sizes=config["roleformer_hidden_dims"],
                      hrr_dim=config["HRR_DIM"],
                      pool0_dim=config["POOLS"][0],
                      roleformer_pool_dims=config["POOLS"][1:],
                      bert_dim=768,
                      rng=seeded_np_gen)
rae = rae.to(device)
print(rae)
sum([p.numel() for p in rae.parameters()])
[p.dtype for p in rae.parameters()]
# rae.forward(x=llm_enc, mask=mask, lens=lens)#.shape
# rfp.forward(llm_enc, mask, lens, syms, episodes=True)

RoleAutoEncoder(
  (role_former_pipeline): RoleFormerPipeline(
    (FC): Linear(in_features=768, out_features=91, bias=True)
    (softmax): Softmax(dim=-1)
    (role_formers): ModuleList(
      (0): RoleFormer(
        (Wk): Linear(in_features=1024, out_features=100, bias=True)
        (Wq): Linear(in_features=1024, out_features=100, bias=True)
        (Wv): Linear(in_features=1024, out_features=56, bias=True)
        (softmax): Softmax(dim=-1)
        (logsoftmax): LogSoftmax(dim=-1)
        (bind_sum): BindSum()
      )
      (1): RoleFormer(
        (Wk): Linear(in_features=1024, out_features=75, bias=True)
        (Wq): Linear(in_features=1024, out_features=75, bias=True)
        (Wv): Linear(in_features=1024, out_features=36, bias=True)
        (softmax): Softmax(dim=-1)
        (logsoftmax): LogSoftmax(dim=-1)
        (bind_sum): BindSum()
      )
    )
  )
  (FC): Linear(in_features=1024, out_features=768, bias=True)
  (gelu): GELU(approximate='none')
  (layer_norm): LayerNorm((

[torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32]

In [131]:
# Load best checkpoint
rae, trloss, epoch = load_model("/content/drive/MyDrive/ICCM26_Data/rae_model_save_3826_0.68trloss.pt", rae)

# Train

## Autoencoder

In [ ]:
class CombinedLoss(nn.Module):
  def __init__(self, reduction, entropy_wt, mse_wt):
    super().__init__()
    self.entropy_wt = entropy_wt
    self.mse_wt = mse_wt
    self.mse_loss = nn.MSELoss(reduction="mean")
  def forward(self, x_o, x_r, entropy, verbose=False):
    this_mse = self.mse_loss(x_o, x_r)
    weighted_mse = self.mse_wt * self.mse_loss(x_o, x_r)
    weighted_entropy = self.entropy_wt * entropy
    if verbose:
      print(f"Unweighted MSE Loss: {this_mse}, Unweighted Entropy: {entropy}")
      print(f"Weighted MSE: {weighted_mse}, Weighted Entropy: {weighted_entropy}")
    out = weighted_mse + weighted_entropy
    return out

In [ ]:
optimizer = torch.optim.AdamW(rae.parameters(), config['learning_rate'])
loss_criterion = CombinedLoss(reduction="mean",
                    entropy_wt=config["entropy_loss_weight"],
                    mse_wt=config["MSE_loss_weight"])


In [ ]:
# save_model(rae, optimizer, -1, 3, "/content/drive/MyDrive/ICCM26_Data/test_model_save.pt")

Free up memory before training.

In [ ]:
if dbert_model:
  del dbert_model
if dbert_tokenizer:
  del dbert_tokenizer
gc.collect()
if device == 'cuda':
  torch.cuda.empty_cache()
rae.train()

In [ ]:
best_loss = float("inf")
# torch.autograd.set_detect_anomaly(True)
for i in range(config["epochs"]):

  total_loss = 0.0
  for b, (batch, att_mask, lens) in enumerate(full_pre_dl):
    optimizer.zero_grad()
    # print(len(batch))
    # print(att_mask)
    # print(lens)
    batch = batch.to(device)
    att_mask = att_mask.to(device)
    lens = torch.tensor(lens).to(device)
    reconstructed_batch, entropy = rae(x=batch, mask=att_mask, lens=lens)
    verbose = (b==0 or b ==1)
    loss = loss_criterion(batch * att_mask.unsqueeze(-1),
                          reconstructed_batch * att_mask.unsqueeze(-1), entropy,
                          verbose=verbose)
    total_loss += loss.item()
    if b % 10 == 0:
      print(f"Batch {b+1}/{len(full_pre_dl)}, {loss}")
    loss.backward()
    optimizer.step()
    del batch, att_mask, lens
    gc.collect()
    if device == 'cuda':
      torch.cuda.empty_cache()
  print(f"Epoch {i+1}/{config['epochs']}: {total_loss}")
  if total_loss < best_loss:
    best_loss = total_loss
    save_model(rae, optimizer, i, best_loss, "/content/drive/MyDrive/ICCM26_Data/rae_model_save.pt")
    print("Saving model")


In [ ]:
0.00043588245171122253 * 768

## FWP with episodes and queries

# Test / Validate

In [68]:
# Load best checkpoint
rae, trloss, epoch = load_model("/content/drive/MyDrive/ICCM26_Data/rae_model_save_3826_0.68trloss.pt", rae)

In [ ]:
def eval_model(model, dataloader):
  best_loss = float("inf")
  # torch.autograd.set_detect_anomaly(True)
  total_loss = 0.0
  for b, (batch, att_mask, lens) in enumerate(dataloader):
    # print(len(batch))
    # print(att_mask)
    # print(lens)
    batch = batch.to(device)
    att_mask = att_mask.to(device)
    lens = torch.tensor(lens).to(device)
    with torch.inference_mode():
      reconstructed_batch, entropy = model(x=batch, mask=att_mask, lens=lens)
    verbose = (b==0 or b ==1)
    loss = loss_criterion(batch * att_mask.unsqueeze(-1),
                          reconstructed_batch * att_mask.unsqueeze(-1), entropy,
                          verbose=verbose)
    total_loss += loss.item()
    if b % 10 == 0:
      print(f"Batch {b+1}/{len(dataloader)}, {loss}")
    del batch, att_mask, lens
    gc.collect()
    if device == 'cuda':
      torch.cuda.empty_cache()
  total_loss /= len(dataloader)
  print(f"Average total loss: {total_loss}")
  return total_loss


In [ ]:
eval_model(rae, full_pre_dl_train)

Unweighted MSE Loss: 0.01984710805118084, Unweighted Entropy: 3.5090528399450704e-05
Weighted MSE: 0.01984710805118084, Weighted Entropy: 1.0527159247430973e-05
Batch 1/45, 0.019857635721564293
Unweighted MSE Loss: 0.016120586544275284, Unweighted Entropy: 4.086215631105006e-05
Weighted MSE: 0.016120586544275284, Weighted Entropy: 1.2258647075213958e-05
Batch 11/45, 0.019181357696652412
Batch 21/45, 0.015226845629513264
Batch 31/45, 0.013587129302322865
Batch 41/45, 0.01509478036314249
Average total loss: 0.016410509476231204


0.016410509476231204

In [ ]:
eval_model(rae, full_pre_dl_val)

Unweighted MSE Loss: 0.014697182923555374, Unweighted Entropy: 8.961466664914042e-05
Weighted MSE: 0.014697182923555374, Weighted Entropy: 2.6884401449933648e-05
Batch 1/6, 0.0147240674123168
Unweighted MSE Loss: 0.011895296163856983, Unweighted Entropy: 7.775570702506229e-05
Weighted MSE: 0.011895296163856983, Weighted Entropy: 2.332671283511445e-05
Average total loss: 0.017542640833805006


0.017542640833805006

In [ ]:
eval_model(rae, full_pre_dl_test)

Unweighted MSE Loss: 0.02059176005423069, Unweighted Entropy: 5.630632949760184e-05
Weighted MSE: 0.02059176005423069, Weighted Entropy: 1.6891899576876312e-05
Batch 1/6, 0.020608652383089066
Unweighted MSE Loss: 0.0246810894459486, Unweighted Entropy: 5.672195402439684e-05
Weighted MSE: 0.0246810894459486, Weighted Entropy: 1.7016587662510574e-05
Average total loss: 0.022398768613735836


0.022398768613735836

# Cog Eval

In [ ]:
# inp = dbert_tokenizer(["This is not a sentence.", "Or is it?"],
#                       padding=True,
#                       return_tensors='pt')
# inp = inp.to(device)
# print(inp)
# print(inp["input_ids"].shape)
# out = dbert_model.forward(inp["input_ids"])
# out.last_hidden_state.shape

{'input_ids': tensor([[ 101, 1188, 1110, 1136,  170, 5650,  119,  102],
        [ 101, 2926, 1110, 1122,  136,  102,    0,    0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0]], device='cuda:0')}
torch.Size([2, 8])


torch.Size([2, 8, 768])

In [54]:
rae.role_former_pipeline

RoleFormerPipeline(
  (FC): Linear(in_features=768, out_features=91, bias=True)
  (softmax): Softmax(dim=-1)
  (role_formers): ModuleList(
    (0): RoleFormer(
      (Wk): Linear(in_features=1024, out_features=100, bias=True)
      (Wq): Linear(in_features=1024, out_features=100, bias=True)
      (Wv): Linear(in_features=1024, out_features=56, bias=True)
      (softmax): Softmax(dim=-1)
      (logsoftmax): LogSoftmax(dim=-1)
      (bind_sum): BindSum()
    )
    (1): RoleFormer(
      (Wk): Linear(in_features=1024, out_features=75, bias=True)
      (Wq): Linear(in_features=1024, out_features=75, bias=True)
      (Wv): Linear(in_features=1024, out_features=36, bias=True)
      (softmax): Softmax(dim=-1)
      (logsoftmax): LogSoftmax(dim=-1)
      (bind_sum): BindSum()
    )
  )
)

In [ ]:

# def text_to_merged_hrr(single_text, ret_hrr="rolesum"):
#   inp = dbert_tokenizer(single_text,
#                         padding=True,
#                         return_tensors='pt')
#   inp = inp.to(device)
#   # print(inp)
#   # print(inp["input_ids"].shape)
#   out = dbert_model.forward(inp["input_ids"])
#   toks, vecs = merge_tokens(inp["input_ids"][0], out.last_hidden_state[0])
#   rolesum, multi, entropy = rae.role_former_pipeline(x=vecs,
#                           mask=torch.ones((len(vecs),)),
#                           lens=torch.sum(inp.attention_mask, dim=-1),
#                           episodes=True)
#   print("Entropy:", entr)
#   # print(inp["input_ids"].shape)
#   # print_tokens(inp["input_ids"][0])
#   if ret_hrr == "rolesum":
#     return rolesum
#   else:
#     return multi
# text_to_merged_bert("Hello there")

(['[CLS]', 'Hello', 'there', '[SEP]'],
 tensor([[ 0.3682,  0.0331,  0.0525,  ..., -0.0586,  0.3385, -0.1074],
         [ 0.3288,  0.1339,  0.1970,  ...,  0.2927,  0.4639,  0.1477],
         [ 0.5269,  0.0162,  0.2196,  ..., -0.2448,  0.2140, -0.2287],
         [ 0.3347,  0.1122,  0.1493,  ..., -0.1299,  1.2507, -0.3033]],
        device='cuda:0'))

## Conjunction Fallacy

In [55]:
conj_text_ds = TextDataset(["Linda is 31 years old, single, outspoken and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and also participated in anti-nuclear demonstrations.",
                            "feminist", "bankteller"])
# print(conj_text_ds[0])
conj_enc_toks, conj_enc_vecs = ds_to_enc(conj_text_ds)
print(conj_enc_toks)
# print(conj_enc_vecs[0])
conj_pre_ds = PreProcessedTextDataset(conj_enc_vecs)
conj_pre_dl = torch.utils.data.DataLoader(
    dataset = conj_pre_ds,
    batch_size=len(conj_text_ds),
    num_workers=1,
    shuffle=False,
    collate_fn=conj_pre_ds.collate_fn
)
rolesum, multi = None, None
for batch, att_mask, lens in conj_pre_dl:
  # print(batch)
  # print(att_mask)
  # print(lens)
  with torch.inference_mode():
    rolesum, multi, entropy = rae.role_former_pipeline(batch.to(device),
                                                      att_mask.to(device),
                                                      torch.tensor(lens).to(device),
                                                      episodes=True)
  print("Entropy:", entropy)
print(rolesum.shape, multi.shape)

[['[CLS]', 'Linda', 'is', '31', 'years', 'old', ',', 'single', ',', 'outspoken', 'and', 'very', 'bright', '.', 'She', 'majored', 'in', 'philosophy', '.', 'As', 'a', 'student', ',', 'she', 'was', 'deeply', 'concerned', 'with', 'issues', 'of', 'discrimination', 'and', 'social', 'justice', ',', 'and', 'also', 'participated', 'in', 'anti', '-', 'nuclear', 'demonstrations', '.', '[SEP]'], ['[CLS]', 'feminist', '[SEP]'], ['[CLS]', 'bankteller', '[SEP]']]
Entropy: tensor(4.2920e-05, device='cuda:0')
torch.Size([3, 45, 1024]) torch.Size([3, 1024])


In [75]:
conj_enc_toks[0][1]

'Linda'

In [56]:
linda_rs = rolesum[0, 1, :]
feminist_rs = rolesum[1, 1, :]
bankteller_rs = rolesum[2, 1, :]
print(linda_rs.shape)

torch.Size([1024])


In [57]:
print(F.cosine_similarity(linda_rs, feminist_rs+bankteller_rs, dim=-1))
print(F.cosine_similarity(linda_rs, bankteller_rs, dim=-1))
print(F.cosine_similarity(feminist_rs, bankteller_rs, dim=-1))


tensor(0.8683, device='cuda:0')
tensor(0.8661, device='cuda:0')
tensor(0.9972, device='cuda:0')


In [2]:
0.8683 - 0.8661

0.0021999999999999797

In [78]:
# multi.shape
linda_multi, feminist_multi, bankteller_multi = multi[0], multi[1], multi[2]
print(F.cosine_similarity(linda_multi, feminist_multi + bankteller_multi, dim=-1))
print(F.cosine_similarity(linda_multi, bankteller_multi, dim=-1))

tensor(0.6700, device='cuda:0')
tensor(0.6632, device='cuda:0')


In [58]:
rae.role_former_pipeline.layer_symbs

tensor([[ 0.0281,  0.0413, -0.0250,  ..., -0.0039,  0.0372,  0.0470],
        [ 0.0549,  0.0322, -0.0110,  ..., -0.0068, -0.0160, -0.0065]],
       device='cuda:0')

In [79]:
multi_l1 = circular_conv(multi, approx_inverse(rae.role_former_pipeline.layer_symbs[1]))
print(F.cosine_similarity(multi_l1[0], multi_l1[1] + multi_l1[2], dim=-1))
print(F.cosine_similarity(multi_l1[0], multi_l1[2], dim=-1))

tensor(0.6700, device='cuda:0')
tensor(0.6632, device='cuda:0')


In [62]:
0.6700 - 0.6632

0.006800000000000028

Why are these returning the same thing? There should be two different levels. Trying out a name

In [60]:
conj_text_ds = TextDataset(["Kathleen is 31 years old, single, outspoken and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and also participated in anti-nuclear demonstrations.",
                            "feminist", "bankteller"])
# print(conj_text_ds[0])
conj_enc_toks, conj_enc_vecs = ds_to_enc(conj_text_ds)
print(conj_enc_toks)
# print(conj_enc_vecs[0])
conj_pre_ds = PreProcessedTextDataset(conj_enc_vecs)
conj_pre_dl = torch.utils.data.DataLoader(
    dataset = conj_pre_ds,
    batch_size=len(conj_text_ds),
    num_workers=1,
    shuffle=False,
    collate_fn=conj_pre_ds.collate_fn
)
rolesum, multi = None, None
for batch, att_mask, lens in conj_pre_dl:
  # print(batch)
  # print(att_mask)
  # print(lens)
  with torch.inference_mode():
    rolesum, multi, entropy = rae.role_former_pipeline(batch.to(device),
                                                      att_mask.to(device),
                                                      torch.tensor(lens).to(device),
                                                      episodes=True)
  print("Entropy:", entropy)
print(rolesum, multi)
kathleen_rs = rolesum[0, 1, :]
feminist_rs = rolesum[1, 1, :]
bankteller_rs = rolesum[2, 1, :]
print(kathleen_rs.shape)
print(F.cosine_similarity(kathleen_rs, feminist_rs+bankteller_rs, dim=-1))
print(F.cosine_similarity(kathleen_rs, bankteller_rs, dim=-1))
print(F.cosine_similarity(feminist_rs, bankteller_rs, dim=-1))
kathleen_multi, feminist_multi, bankteller_multi = multi[0], multi[1], multi[2]
print(F.cosine_similarity(kathleen_multi, feminist_multi + bankteller_multi, dim=-1))
print(F.cosine_similarity(kathleen_multi, bankteller_multi, dim=-1))

[['[CLS]', 'Kathleen', 'is', '31', 'years', 'old', ',', 'single', ',', 'outspoken', 'and', 'very', 'bright', '.', 'She', 'majored', 'in', 'philosophy', '.', 'As', 'a', 'student', ',', 'she', 'was', 'deeply', 'concerned', 'with', 'issues', 'of', 'discrimination', 'and', 'social', 'justice', ',', 'and', 'also', 'participated', 'in', 'anti', '-', 'nuclear', 'demonstrations', '.', '[SEP]'], ['[CLS]', 'feminist', '[SEP]'], ['[CLS]', 'bankteller', '[SEP]']]
Entropy: tensor(4.2978e-05, device='cuda:0')
tensor([[[-3.8434e+00, -8.6269e+00, -1.9343e+01,  ...,  8.9243e+00,
           1.3506e+00,  2.7286e+00],
         [-9.8069e+00,  4.2735e+00, -5.5353e+00,  ...,  1.4042e-01,
          -5.2886e+00, -8.0151e+00],
         [-1.0670e+01,  3.9871e+00, -7.3985e+00,  ..., -1.2477e+00,
          -6.2457e+00, -8.8698e+00],
         ...,
         [-1.0781e+01,  3.5934e+00, -7.8430e+00,  ..., -4.1702e-01,
          -5.3942e+00, -9.5667e+00],
         [-1.2688e+01,  4.5691e+00, -1.1061e+01,  ...,  4.4070e+0

In [61]:
0.8687-0.8669

0.0018000000000000238

## IAT Spot Check

In [ ]:
iat_spot_words = ["black", "Black", "white",
                  "White", "pleasant", "unpleasant"]
word2idx = {w:i for i, w in enumerate(iat_spot_words)}
word2idx

{'black': 0,
 'Black': 1,
 'white': 2,
 'White': 3,
 'pleasant': 4,
 'unpleasant': 5}

In [ ]:
iat_spot_text_ds = TextDataset(iat_spot_words)
# print(conj_text_ds[0])
iat_spot_enc_toks, iat_spot_enc_vecs = ds_to_enc(iat_spot_text_ds)
print(iat_spot_enc_toks)
# print(conj_enc_vecs[0])
iat_spot_pre_ds = PreProcessedTextDataset(iat_spot_enc_vecs)
iat_spot_pre_dl = torch.utils.data.DataLoader(
    dataset = iat_spot_pre_ds,
    batch_size=len(iat_spot_text_ds),
    num_workers=1,
    shuffle=False,
    collate_fn=iat_spot_pre_ds.collate_fn
)
iat_spot_rolesum, iat_spot_multi = None, None
for batch, att_mask, lens in iat_spot_pre_dl:
  # print(batch)
  # print(att_mask)
  # print(lens)
  with torch.inference_mode():
    iat_spot_rolesum, iat_spot_multi, iat_spot_entropy = rae.role_former_pipeline(batch.to(device),
                                                      att_mask.to(device),
                                                      torch.tensor(lens).to(device),
                                                      episodes=True)
  print("Entropy:", entropy)
print(iat_spot_rolesum, iat_spot_multi)

[['[CLS]', 'black', '[SEP]'], ['[CLS]', 'Black', '[SEP]'], ['[CLS]', 'white', '[SEP]'], ['[CLS]', 'White', '[SEP]'], ['[CLS]', 'pleasant', '[SEP]'], ['[CLS]', 'unpleasant', '[SEP]']]
Entropy: tensor(2.2199e-10, device='cuda:0')
tensor([[[-0.0072, -0.0655, -0.1465,  ...,  0.0390,  0.0085,  0.0796],
         [-0.0305,  0.0477, -0.0350,  ..., -0.0041, -0.0695, -0.0401],
         [-0.0293,  0.0490, -0.0477,  ...,  0.0030, -0.0505, -0.0245]],

        [[-0.0060, -0.0667, -0.1463,  ...,  0.0392,  0.0065,  0.0780],
         [-0.0302,  0.0434, -0.0323,  ..., -0.0059, -0.0642, -0.0398],
         [-0.0348,  0.0491, -0.0533,  ...,  0.0017, -0.0605, -0.0304]],

        [[-0.0061, -0.0657, -0.1471,  ...,  0.0392,  0.0057,  0.0784],
         [-0.0310,  0.0488, -0.0361,  ..., -0.0051, -0.0706, -0.0399],
         [-0.0350,  0.0474, -0.0537,  ...,  0.0090, -0.0567, -0.0232]],

        [[-0.0051, -0.0672, -0.1461,  ...,  0.0393,  0.0048,  0.0769],
         [-0.0315,  0.0434, -0.0328,  ..., -0.0060, -0.0

In [ ]:
word2rs = {w:rolesum[idx, 1, :] for w, idx in word2idx.items()}
word2rs

{'black': tensor([-0.0305,  0.0477, -0.0350,  ..., -0.0041, -0.0695, -0.0401],
        device='cuda:0'),
 'Black': tensor([-0.0302,  0.0434, -0.0323,  ..., -0.0059, -0.0642, -0.0398],
        device='cuda:0'),
 'white': tensor([-0.0310,  0.0488, -0.0361,  ..., -0.0051, -0.0706, -0.0399],
        device='cuda:0'),
 'White': tensor([-0.0315,  0.0434, -0.0328,  ..., -0.0060, -0.0641, -0.0399],
        device='cuda:0'),
 'pleasant': tensor([-0.0357,  0.0481, -0.0361,  ..., -0.0082, -0.0658, -0.0419],
        device='cuda:0'),
 'unpleasant': tensor([-0.0315,  0.0418, -0.0322,  ..., -0.0065, -0.0623, -0.0408],
        device='cuda:0')}

In [ ]:
bu = F.cosine_similarity(word2rs["black"], word2rs["unpleasant"], dim=-1)
# yes there probably is a faster way to do this
for word1 in iat_spot_words:
  for word2 in iat_spot_words:
    wsim = F.cosine_similarity(word2rs[word1], word2rs[word2], dim=-1)
    print(f"Similarity between {word1} and {word2}:", wsim.item())

Similarity between black and black: 1.0000001192092896
Similarity between black and Black: 0.9980423450469971
Similarity between black and white: 0.9994730353355408
Similarity between black and White: 0.9980441331863403
Similarity between black and pleasant: 0.9978687763214111
Similarity between black and unpleasant: 0.996900200843811
Similarity between Black and black: 0.9980423450469971
Similarity between Black and Black: 1.0
Similarity between Black and white: 0.9972985982894897
Similarity between Black and White: 0.9998810291290283
Similarity between Black and pleasant: 0.99747633934021
Similarity between Black and unpleasant: 0.9993751645088196
Similarity between white and black: 0.9994730353355408
Similarity between white and Black: 0.9972985982894897
Similarity between white and white: 1.0
Similarity between white and White: 0.9976005554199219
Similarity between white and pleasant: 0.9977967739105225
Similarity between white and unpleasant: 0.9959453344345093
Similarity between 

In [ ]:
# round 5 - black/pleasant + white/unpleasant
round5sim = 0.5*(0.9978687763214111 + 0.9959453344345093)
# round 3 - white/pleasant + black/unpleasant
round3sim = 0.5*(0.9977967739105225 + 0.996900200843811)
print(round3sim, round5sim)

0.9973484873771667 0.9969070553779602


In [ ]:
# modified from original
model2_F, model2_f, model2_rt = 1.219, 0.02, -2
def sim_to_rt(sim, F, f):
  A = math.log((sim**2)/(1-sim**2)) # from HDM paper
  print(A)
  print(A > model2_rt)
  return F * math.exp(-f * A)
print(sim_to_rt(round3sim, model2_F, model2_f) * 1000)
print(sim_to_rt(round5sim, model2_F, model2_f) * 1000)

5.235494387886397
True
1097.8140361299231
5.080836703178544
True
1101.2150008129238


## Episodes

In [51]:
all_interviews.keys()

dict_keys(['christia_adair.txt', 'elizabeth_cardozo_baker.txt', 'etta_moten_barnett.txt', 'frances_albrier.txt', 'frankie_adams.txt', 'jessie_abbot.txt', 'kathleen_adams.txt', 'margaret_walker_alexander.txt', 'norma_boyd (1).txt', 'norma_boyd.txt', 'sadie_alexander.txt'])

In [ ]:
# only select episodes with "black" or "Black" in them
episode_keys = ['christia_adair.txt', 'elizabeth_cardozo_baker.txt',
            'frances_albrier.txt', 'frankie_adams.txt', 'jessie_abbot.txt',
            'margaret_walker_alexander.txt', 'sadie_alexander.txt']
episodes_text = []
for k in episode_keys:
  episodes_text.extend(all_interviews[k])
episodes_text

In [111]:
episodes_text_ds = TextDataset(episodes_text)
epis_merged_toks, epis_merged_vecs, epis_merged_syms, word2sym = ds_to_enc(episodes_text_ds, symbols=True)

In [69]:
print(len(epis_merged_vecs), epis_merged_vecs[0].shape)
len(epis_merged_syms), epis_merged_syms[0].shape

4058 torch.Size([4, 768])


(4058, torch.Size([4, 1024]))

Add the symbol for "black" and "Black" (should not have made BERT case sensitive in hindsight) to vectors corresponding to tokens with "I", "me", and "my" (and case sensitive variants).

In [107]:
first_person_pronouns = {"I", "Me", "me", "My", "my"}
sym_blk = 0.5 * (word2sym["Black"] + word2sym["black"])

In [171]:
sym_white = 0.5 * (word2sym["white"] + word2sym["White"])

In [113]:
print("Symbol Before", epis_merged_syms[9][1])

Symbol Before tensor([ 0.0283, -0.0174, -0.0441,  ...,  0.0069, -0.0063,  0.0074])


In [114]:
for tok_list, stacked_syms in zip(epis_merged_toks, epis_merged_syms):
  assert len(tok_list) == stacked_syms.size(0)
  matched_inds = [i for i in range(len(tok_list)) if tok_list[i] in first_person_pronouns]
  print(matched_inds)
  for ind in matched_inds:
    # S x N indexed to N
    epis_merged_syms[ind] += sym_blk

[]
[]
[]
[]
[]
[]
[]
[]
[]
[1, 12, 18]
[]
[7, 12]
[2, 10]
[2]
[]
[]
[4, 9, 35]
[12]
[]
[]
[]
[]
[]
[]
[]
[]
[9]
[7]
[]
[]
[]
[1, 6]
[3, 8, 14]
[]
[]
[]
[]
[]
[12]
[15]
[10]
[]
[]
[4, 8, 13]
[5, 15]
[]
[2, 7, 15]
[]
[]
[]
[]
[4, 7, 11]
[]
[]
[11]
[2, 7]
[1]
[10]
[]
[18]
[]
[10]
[1, 4, 11]
[11]
[12]
[]
[]
[5, 10]
[5]
[13]
[]
[]
[]
[]
[3, 7]
[]
[]
[]
[]
[]
[]
[]
[]
[3, 6]
[10, 12]
[3]
[]
[3, 15]
[10]
[5, 8]
[1, 6, 10, 13, 19]
[5, 12]
[2]
[31]
[]
[]
[]
[]
[5, 9, 15]
[]
[14]
[5, 15]
[]
[2]
[11]
[1, 31]
[8]
[4, 12]
[]
[5, 7]
[9, 12]
[1, 9, 11, 20]
[3, 12, 17]
[6, 9]
[2]
[]
[5]
[]
[]
[]
[]
[3]
[4, 12]
[3, 9]
[1, 4, 11]
[5, 13, 18]
[]
[2, 13]
[2, 11]
[8]
[7, 10]
[5, 14, 16]
[]
[10]
[1, 12]
[3, 10]
[10, 16]
[11]
[14, 22]
[8]
[]
[9]
[]
[2]
[]
[]
[]
[]
[]
[]
[12, 16]
[]
[]
[5, 12]
[12]
[]
[]
[]
[]
[13]
[]
[3, 15]
[7, 12]
[6]
[15]
[9]
[]
[3, 18]
[8, 13]
[]
[6]
[1]
[8, 13]
[6]
[1, 11]
[2, 12, 23]
[]
[4, 15]
[8, 11]
[15]
[]
[1, 12, 14]
[4]
[1, 7, 17]
[1]
[]
[4, 9]
[2, 15]
[14]
[2, 7]
[5, 9]
[]
[]
[]

In [115]:
print("Symbol After", epis_merged_syms[9][1])

Symbol After tensor([ 2.4849, -3.4139, -1.4252,  ...,  6.9068, -1.9111,  1.8124])


In [116]:
episodes_pre_ds = PreProcessedTextDataset(vecs=epis_merged_vecs,
                                          syms=epis_merged_syms)
episodes_pre_dl = torch.utils.data.DataLoader(
    dataset=episodes_pre_ds,
    batch_size=config["rf_batch_size"],
    shuffle=False,
    collate_fn=episodes_pre_ds.collate_fn
)

In [153]:
multi_combined = torch.zeros(config['HRR_DIM']).to(device)
for batch in episodes_pre_dl:
  vecs_padded, att_mask, lens, syms_padded = batch
  print(vecs_padded.shape, att_mask.shape, len(lens), syms_padded.shape)
  print(vecs_padded[0, :, 0], att_mask[0], lens[0], syms_padded[0, :, 0])

  with torch.inference_mode():
    rolesum, multi, entropy = rae.role_former_pipeline(vecs_padded.to(device),
                                                      att_mask.to(device),
                                                      torch.tensor(lens).to(device),
                                                      episodes=True,
                                                      symbs=syms_padded.to(device))
    print(multi.shape)
    multi_combined += torch.sum(multi, dim=0) / torch.tensor(sum(lens)).to(device)


torch.Size([100, 39, 768]) torch.Size([100, 39]) 100 torch.Size([100, 39, 1024])
tensor([0.4833, 0.0545, 0.3013, 1.1899, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000]) tensor([1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.]) 4 tensor([ 0.0048, -0.0129,  0.0101, -0.0118,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  

In [155]:
multi_combined_L1 = circular_conv(multi_combined, approx_inverse(rae.role_former_pipeline.layer_symbs[0]))

In [172]:
blackl1 = circular_conv(multi_combined_L1,  approx_inverse(sym_blk.to(device)))
whitel1 = circular_conv(multi_combined_L1,  approx_inverse(sym_white.to(device)))

In [157]:
blackl1

tensor([  4.8069,   0.7582, -13.9700,  ..., -19.6981,  24.7412, -19.8760],
       device='cuda:0')

In [146]:
multi_combined.shape, word2sym['DRR'].shape

(torch.Size([1024]), torch.Size([1024]))

In [152]:
F.cosine_similarity(multi_combined.to('cpu') / sum(lens[:config['rf_batch_size']]),
                    , dim=-1)

tensor(-0.0454)

In [ ]:
F.cosine_similarity(blackl1, )

Get Individual Words

In [159]:
adj_text_ds = TextDataset(["pleasant", "unpleasant"])
# print(conj_text_ds[0])
adj_merged_toks, adj_merged_vecs, adj_merged_syms, adj_word2sym = ds_to_enc(adj_text_ds, symbols=True)
# print(conj_enc_toks)
# print(conj_enc_vecs[0])
adj_pre_ds = PreProcessedTextDataset(adj_merged_vecs, adj_merged_syms)
adj_pre_dl = torch.utils.data.DataLoader(
    dataset = adj_pre_ds,
    batch_size=len(adj_pre_ds),
    num_workers=1,
    shuffle=False,
    collate_fn=adj_pre_ds.collate_fn
)
adj_rolesum, adj_multi = None, None
for batch in adj_pre_dl:
  vecs_padded, att_mask, lens, syms_padded = batch
  # print(batch)
  # print(att_mask)
  # print(lens)
  adj_rolesum, adj_multi, adj_entropy = rae.role_former_pipeline(vecs_padded.to(device),
                                                      att_mask.to(device),
                                                      torch.tensor(lens).to(device),
                                                      episodes=True,
                                                      symbs=syms_padded.to(device))
  print("Entropy:", adj_entropy)
print(adj_rolesum.shape, adj_multi.shape)

Entropy: tensor(5.0450e-10, device='cuda:0', grad_fn=<DivBackward0>)
torch.Size([2, 3, 1024]) torch.Size([2, 1024])


In [161]:
adj_word2sym

{'unpleasant': tensor([-0.0438, -0.0375,  0.0595,  ...,  0.0335,  0.0106,  0.0282]),
 '[SEP]': tensor([-0.0002, -0.0247, -0.0023,  ...,  0.0348, -0.0095,  0.0116]),
 '[CLS]': tensor([ 0.0243, -0.0348, -0.0052,  ...,  0.0372, -0.0012, -0.0296]),
 'pleasant': tensor([-0.0446,  0.0030, -0.0702,  ...,  0.0062,  0.0030, -0.0098])}

In [160]:
adj_multi_L1 = circular_conv(adj_multi[0], approx_inverse(rae.role_former_pipeline.layer_symbs[0]))

In [164]:
pleasantL1 = circular_conv(adj_multi_L1,  approx_inverse(adj_word2sym["pleasant"].to(device)))
unpleasantL1 = circular_conv(adj_multi_L1,  approx_inverse(adj_word2sym["unpleasant"].to(device)))

### IAT Fitting

In [165]:
# round 3 - white/pleasant + black/unpleasant
print(F.cosine_similarity(blackl1, pleasantL1, dim=-1))
print(F.cosine_similarity(blackl1, unpleasantL1, dim=-1))

tensor(0.3579, device='cuda:0', grad_fn=<SumBackward1>)
tensor(-0.0489, device='cuda:0', grad_fn=<SumBackward1>)


In [173]:
# round 5 - black/pleasant + white/unpleasant
print(F.cosine_similarity(whitel1, pleasantL1, dim=-1))
print(F.cosine_similarity(whitel1, unpleasantL1, dim=-1))

tensor(-0.0267, device='cuda:0', grad_fn=<SumBackward1>)
tensor(0.0008, device='cuda:0', grad_fn=<SumBackward1>)


In [177]:
# round 3 - white/pleasant + black/unpleasant
round3sim = 0.5 * (F.cosine_similarity(whitel1, pleasantL1, dim=-1) + F.cosine_similarity(blackl1, unpleasantL1, dim=-1))
# round 5 - black/pleasant + white/unpleasant
round5sim = 0.5 * (F.cosine_similarity(blackl1, pleasantL1, dim=-1) + F.cosine_similarity(whitel1, unpleasantL1, dim=-1))
round3sim = round3sim.item()
round5sim = round5sim.item()
print(round3sim, round5sim)

-0.03776571899652481 0.17935015261173248


In [180]:
def sim_to_rt(sim, F, f):
  A = math.log((sim**2)/(1-sim**2)) # from HDM paper
  print(A)
  print(A > model2_rt)
  return F * math.exp(-f * A)

In [194]:
model2_F, model2_f, model2_rt = 0.9, 0.025, -7
# sim_to_rt(sim, F, f) -> F * math.exp(-f * A)
print(sim_to_rt(round3sim, model2_F, model2_f) * 1000)
print(sim_to_rt(round5sim, model2_F, model2_f) * 1000)

-6.551279717801587
True
1060.1617356506547
-3.4041352610593254
True
979.9466631739427


In [ ]:
# 1051 960